# Feature experiment: OSF criterion

Test whether a feature based on tool wear and torque can improve machine-failure prediction.

## Load prepared data

Load the raw train and test features created by the data-preparation step.

In [1]:
import pandas as pd

X_train_raw = pd.read_csv(
    "../data/processed/X_train_raw.csv",
    index_col="record_index",
)
X_test_raw = pd.read_csv(
    "../data/processed/X_test_raw.csv",
    index_col="record_index",
)

y_train = pd.read_csv(
    "../data/processed/y_train.csv",
    index_col="record_index",
).squeeze("columns")
y_test = pd.read_csv(
    "../data/processed/y_test.csv",
    index_col="record_index",
).squeeze("columns")


## Validate the loaded data

Check that feature and target indices still match before creating a new feature.

In [2]:
assert X_train_raw.index.equals(y_train.index)
assert X_test_raw.index.equals(y_test.index)
assert "record_index" not in X_train_raw.columns
assert "record_index" not in X_test_raw.columns

print("Train shape:", X_train_raw.shape)
print("Test shape:", X_test_raw.shape)


Train shape: (8000, 8)
Test shape: (2000, 8)


## Add the OSF criterion

Create the feature from measurements available before a failure occurs. This is an exploratory feature experiment, so the main baseline remains unchanged.

In [3]:
OSF_FEATURE = "OSF criterion"

X_train_osf = X_train_raw.copy()
X_test_osf = X_test_raw.copy()

for X in (X_train_osf, X_test_osf):
    X[OSF_FEATURE] = X["Tool wear [min]"] * X["Torque [Nm]"]

print(X_train_osf[["Tool wear [min]", "Torque [Nm]", OSF_FEATURE]].head())
print("New train shape:", X_train_osf.shape)
print("New test shape:", X_test_osf.shape)


              Tool wear [min]  Torque [Nm]  OSF criterion
record_index                                             
3693                      202         31.5         6363.0
590                       233         36.7         8551.1
6770                       25         40.1         1002.5
1412                      194         53.2        10320.8
3298                       61         38.1         2324.1
New train shape: (8000, 9)
New test shape: (2000, 9)


## Prepare feature variants

Create a baseline dataset and a dataset with the additional OSF criterion.

In [5]:
BASE_FEATURES = X_train_raw.columns.tolist()

X_train_baseline = X_train_raw.copy()
X_test_baseline = X_test_raw.copy()

X_train_osf = X_train_raw.copy()
X_test_osf = X_test_raw.copy()

X_train_osf['OSF criterion'] = (X_train_osf['Tool wear [min]'] * X_train_osf['Torque [Nm]'])
X_test_osf['OSF criterion'] = (X_test_osf['Tool wear [min]'] * X_test_osf['Torque [Nm]'])

print("Baseline features:", X_train_baseline.shape[1])
print("OSF experiment features:", X_train_osf.shape[1])
print("New feature:", "OSF criterion" in X_train_osf.columns)

Baseline features: 8
OSF experiment features: 9
New feature: True


## Create stratified cross-validation splits



Use the same class proportions in every validation fold.

In [9]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold_number, (train_indices, validation_indices) in enumerate(cv.split(X_train_baseline, y_train), start = 1):
    y_fold_train = y_train.iloc[train_indices]
    y_fold_validation = y_train.iloc[validation_indices]

    print(
        f"Fold {fold_number}: "
        f"train={len(train_indices)}, "
        f"validation={len(validation_indices)}, "
        f"failures in train={y_fold_train.sum()}, "
        f"failures in validation={y_fold_validation.sum()}"
    )

Fold 1: train=6400, validation=1600, failures in train=212, failures in validation=52
Fold 2: train=6400, validation=1600, failures in train=211, failures in validation=53
Fold 3: train=6400, validation=1600, failures in train=211, failures in validation=53
Fold 4: train=6400, validation=1600, failures in train=211, failures in validation=53
Fold 5: train=6400, validation=1600, failures in train=211, failures in validation=53


## Define models and evaluation metrics

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

models = {
    "Logistic Regression": LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42,
    ),
    "Random Forest": RandomForestClassifier(
        class_weight='balanced',
        n_estimators=200,
        random_state=42,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42,
    ),
}


## Evaluate models with cross-validation

In [ ]:
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler

DECISION_THRESHOLD = 0.5
CATEGORICAL_FEATURES = ["Type_L", "Type_M"]

def prepare_fold_data(X_train_fold, X_validation_fold):
    """
    Scale numeric features using only the training part of one fold.
    """
    X_train_fold = X_train_fold.copy()
    X_validation_fold = X_validation_fold.copy()

    numeric_features = [
        column
        for column in X_train_fold.columns
        if column not in CATEGORICAL_FEATURES
    ]

    scaler = StandardScaler()

    X_train_fold[numeric_features] = scaler.fit_transform(X_train_fold[numeric_features])
    X_validation_fold[numeric_features] = scaler.transform(X_validation_fold[numeric_features])

    return X_train_fold, X_validation_fold

def get_metrics(y_true, y_proba):
    y_pred = (y_proba >= DECISION_THRESHOLD).astype(int)

    return {
        "accuracy": (y_true == y_pred).mean(),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "average_precision": average_precision_score(y_true, y_proba),
    }

def evaluate_model_with_cv(model, X, y):
    """
    Evaluate one model across all cross-validation folds.
    """
    fold_metrics = []

    for train_indices, validation_indices in cv.split(X, y):
        X_train_fold = X.iloc[train_indices]
        X_validation_fold = X.iloc[validation_indices]

        y_train_fold = y.iloc[train_indices]
        y_validation_fold = y.iloc[validation_indices]

        X_train_fold, X_validation_fold = prepare_fold_data(X_train_fold, X_validation_fold)

    fold_model = clone(model)
    fold_model.fit(X_train_fold, y_train_fold)

    y_proba = fold_model.predict_proba(X_validation_fold)[:, 1]

    fold_metrics.append(calculate_metrics(y_validation_fold, y_proba))

    results = {}

    for metric in fold_metrics_df.columns:
        results[f"{metric}_mean"] = fold_metrics_df[metric].mean()
        results[f"{metric}_std"] = fold_metrics_df[metric].std()
    return results, fold_metrics_df